# 医疗推理指令微调：LoRA 参数高效微调

课程前两个实验（RBT3 情感分类、mengzi-T5 摘要）走的都是全参数微调路线；本实验转向参数高效微调（PEFT）：**冻结基座模型，只用 LoRA 旁路**完成一次完整的医疗推理指令微调（SFT）。

## 实验配置

- **基座模型**：本地 `Qwen3.5-2B`（`../week20/Qwen3.5-2B`；1.88B 文本参数，24 层混合注意力，原生 `<think>` 思考模板）
- **训练数据**：`medical-o1-reasoning-SFT`（中文 `zh` 配置）

## 实验目的

1. **掌握 LoRA 原理与超参数**：在冻结权重 $W$ 旁训练低秩矩阵 $\Delta W = \frac{\alpha}{r}BA$，理解秩 $r$、缩放系数 $\alpha$、`target_modules`（Q/K/V/O 投影）等设计的作用，以及「只训约 0.04% 的参数为什么仍然有效」。
2. **跑通 peft 版 SFT 全流程**：数据加载 → `apply_chat_template` 对话模板拼装 → prompt 段 `-100` 掩码 → `get_peft_model` 注入 → `Trainer` 训练 → adapter 保存 → `merge_and_unload` 合并导出。
3. **掌握 SFT 训练样本的构造方法**：验证「完整序列 + 提问掩码」的训练目标设计（只在 assistant 回复段计算损失），并理清它与推理输入（`add_generation_prompt=True`）的对齐关系。
4. **观察指令微调的效果**：用医疗推理数据集（`medical-o1-reasoning-SFT`）让模型学会「先思考（`<think>` 思维链）、后回答」的行为模式，并观察生成效果。
5. **对照全参微调、认识产物管理**：比较 LoRA 与全参微调在参数量 / 显存 / 训练成本上的差异；分析 checkpoint 与最优模型（`load_best_model_at_end`）、adapter 与合并后完整模型的区别与用途。

## 训练流程

一条流水线串起来：**数据 → 对话模板拼装 → 掩码 labels → 注入 LoRA → Trainer 训练 → 保存 adapter →（可选）合并导出 → 推理演示**。

**本机配置**

| 配置项 | 取值 |
|---|---|
| 基座模型 | `../week20/Qwen3.5-2B`（1.88B 文本参数） |
| 训练数据 | `medical-o1-reasoning-SFT` 中文版前 200 条（160 训练 / 40 评估，seed=42） |
| 对话模板 | prompt 渲染 `enable_thinking=True`，与完整样本前缀对齐 |
| 精度与设备 | bf16（加载与训练）；设备由 Trainer 管理（不用 `device_map`） |
| LoRA | r=8、α=16、dropout=0.05，挂 `q/k/v/o`：6 个全注意力层 × 4 = 24 个挂载点；可训练参数 73.7 万 ≈ 0.039% |
| 批次与步数 | 训练 batch 1 × 累积 32（等效 32；评估 batch 1）→ 5 轮 = 25 步 |
| 优化设置 | 学习率 2e-4、cosine、warmup_steps=3 |
| 显存策略 | 梯度检查点开启；`MAX_LENGTH=1024`（本子集最长 989 token，无截断） |
| 输出目录 | `./results_lora`（checkpoint）、`./lora_adapter`（适配器）、`./lora_merged`（可选合并） |

> ⚠️ 训练格 `trainer.train()` 一旦执行即正式开始训练。

In [1]:
# ==================== 1) 依赖与全局配置 ====================
# 模型与数据均为本机文件，无需联网/镜像

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # 只用 0 号卡（须在导入 torch 前设置）
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"   # 降低显存碎片（6GB 卡易碎）

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel

MODEL_PATH  = "../week20/Qwen3.5-2B"                                  # 基座（1.88B 文本参数）
DATA_PATH   = "medical-o1-reasoning-SFT/medical_o1_sft_Chinese.json"  # 医疗推理数据（中文版）
OUTPUT_DIR  = "./results_lora"     # 训练中间产物（checkpoint）
ADAPTER_DIR = "./lora_adapter"     # 训练结束保存的 LoRA 适配器
MERGED_DIR  = "./lora_merged"      # （可选）合并导出的完整模型

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无（训练需要 GPU）")

d:\project\step3\llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.10.0+cu128
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ==================== 2) 分词器与对话模板演示 ====================

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

demo = [
    {"role": "user", "content": "感冒了该怎么办？"},
    {"role": "assistant", "content": "<think>\n先区分病毒性与细菌性……\n</think>\n建议多休息、多饮水。"},
]

# 完整训练样本：assistant 里的 <think>…</think> 会被 Qwen3.5 模板解析成「思考段 + 答案段」再渲染
full = tokenizer.apply_chat_template(demo, tokenize=False, add_generation_prompt=False)

# 推理 prompt：启用 enable_thinking=True（结尾才是 "<|im_start|>assistant\n<think>\n"，与完整样本前缀对齐）
prompt = tokenizer.apply_chat_template(
    demo[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=True)

print("完整样本结尾 :", repr(full[-42:]))
print("prompt 结尾  :", repr(prompt[-42:]))
print("前缀对齐     :", full.startswith(prompt))
print()
# 若不传 enable_thinking，结尾是空思考块 "<think>\n\n</think>\n\n"——与完整样本对不齐
print("默认渲染结尾 :", repr(
    tokenizer.apply_chat_template(demo[:-1], tokenize=False, add_generation_prompt=True)[-42:]))

完整样本结尾 : '分病毒性与细菌性……\n</think>\n\n建议多休息、多饮水。<|im_end|>\n'
prompt 结尾  : '？<|im_end|>\n<|im_start|>assistant\n<think>\n'
前缀对齐     : True

默认渲染结尾 : '\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


In [3]:
# ==================== 3) 加载基座模型 ====================
# bf16 加载（1.88B ≈ 3.8GB，6GB 显存必须半精度）；
# 训练时不用 device_map="auto"（那是推理/自动分片用法，设备交给 Trainer）

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16)
print("类别:", type(model).__name__,
      "| 参数量: {:.3f} B".format(sum(p.numel() for p in model.parameters()) / 1e9))

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 3047.02it/s]

类别: Qwen3_5ForCausalLM | 参数量: 1.882 B


In [4]:
# ==================== 4) 加载与划分数据 ====================
# 加载中文版 JSON；先取前 200 条跑通流程（去掉切片即全量 ~2 万条）

ds = load_dataset("json", data_files=DATA_PATH, split="train[:200]")

split_dataset = ds.train_test_split(test_size=0.2, seed=42)   # 160 训练 / 40 评估
print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 160
    })
    test: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 40
    })
})


In [5]:
# ==================== 5) 数据预处理：拼对话 + 掩码 labels ====================
MAX_LENGTH = 1024  # 超长截断阈值（本子集实测最长 989 token，暂无截断；显存紧张可再降）

def preprocess_function(examples):
    input_ids_list, labels_list = [], []
    for question, cot, response in zip(examples["Question"], examples["Complex_CoT"], examples["Response"]):

        messages = [
            {"role": "system", "content": "您是一位在临床推理、诊断和治疗计划方面具有高级知识的医学专家。请回答以下医学问题。"},
            {"role": "user", "content": question},
            # 思维链包进 <think>：模板会解析为「思考段 + 答案段」渲染
            {"role": "assistant", "content": f"<think>\n{cot}\n</think>\n{response}"},
        ]

        # 完整样本
        full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

        # prompt 部分（enable_thinking=True，结尾 "<|im_start|>assistant\n<think>\n" 与 full 前缀对齐）
        prompt_text = tokenizer.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=True)
        assert full_text.startswith(prompt_text), "prompt 不是完整样本前缀，检查模板参数！"

        full_ids   = tokenizer.encode(full_text,   add_special_tokens=False)
        prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)

        # 超长：截到 MAX_LENGTH-1 再补 EOS（保证"学会停"）
        if len(full_ids) > MAX_LENGTH:
            full_ids = full_ids[:MAX_LENGTH - 1]
            full_ids.append(tokenizer.eos_token_id)

        # labels：提问段 -100（不算 loss），回答段照原样训练
        labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])[:len(full_ids)]

        input_ids_list.append(full_ids)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": [[1] * len(ids) for ids in input_ids_list],
        "labels": labels_list,
    }

In [6]:
# ==================== 6) 应用到数据集并体检 ====================
processed_dataset = split_dataset.map(
    preprocess_function, batched=True, remove_columns=ds.column_names)
print(processed_dataset)

# 长度体检（本子集实测最长 989 token）
lens = sorted(len(x) for x in processed_dataset["train"]["input_ids"])
n = len(lens)
print()
print(f"训练样本 {n} 条 | token 长度: min {lens[0]} / 中位 {lens[n//2]} / P90 {lens[int(n*0.9)]} / max {lens[-1]}")
print(f"被截断到 {MAX_LENGTH} 的样本: {sum(1 for l in lens if l == MAX_LENGTH)} 条")

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 160
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40
    })
})

训练样本 160 条 | token 长度: min 285 / 中位 512 / P90 678 / max 989
被截断到 1024 的样本: 0 条


In [7]:
# ==================== 7) 注入 LoRA + collator ====================
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()
# 本机实测：24 个挂载点（= 6 个全注意力层 × 4 个投影），可训练参数 737,280 ≈ 0.039%
# （Qwen3.5-2B 是混合注意力：24 层中仅索引 3/7/11/15/19/23 为全注意力；想扩大覆盖面可再加
#  "linear_attn.out_proj"、"mlp.*" 等）

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)

# 冻结基座 + 梯度检查点：输入 embedding 必须能带梯度（详解见 大模型微调.ipynb）
lora_model.enable_input_require_grads()

trainable params: 737,280 || all params: 1,882,562,368 || trainable%: 0.0392


In [8]:
# ==================== 8) 训练参数 ====================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,
    per_device_train_batch_size=1,      # 6GB 显存下稳妥取 1（词表 248k，logits 很占显存）
    gradient_accumulation_steps=32,     # 等效 batch = 1 × 32 = 32 → 仍是 25 步
    per_device_eval_batch_size=1,       # 评估也必须取 1：默认 8 时 8×~850token×248k 词表的 CE 需单块 ~6GB → OOM

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=3,                     # ≈ 25 步的 10%（transformers 5 用 warmup_steps）
    bf16=True,                          # 半精度训练（1.88B 必须）

    save_strategy="steps", save_steps=10, save_total_limit=50,
    eval_strategy="steps",   eval_steps=10,
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,

    gradient_checkpointing=True,
    logging_steps=5,                    # 每 5 步记录 train loss（便于观察）
    report_to="none",                   # 不接 tensorboard / wandb
)

In [9]:
# ==================== 9) Trainer ====================
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    data_collator=data_collator,
)

d:\project\step3\llm\Lib\site-packages\torch\nn\modules\module.py:1367: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


In [10]:
# ==================== 10) 训练并保存适配器 ====================
# ⚠️ 执行本格 = 正式开始训练（160 条 × 5 轮 = 25 步）
if __name__ == "__main__":              # notebook 里恒为 True；兼防多进程子进程重复执行
    trainer.train()
    print("最优 checkpoint:", trainer.state.best_model_checkpoint, "| eval_loss:", trainer.state.best_metric)
    trainer.save_model(ADAPTER_DIR)     # PeftModel → 只保存适配器（adapter_config + adapter_model）
    tokenizer.save_pretrained(ADAPTER_DIR)

Step,Training Loss,Validation Loss
10,2.370035,2.287736
20,2.250996,2.229170
25,2.236830,2.225773


最优 checkpoint: ./results_lora\checkpoint-25 | eval_loss: 2.2257726192474365


In [11]:
# ==================== 11)（可选）合并适配器 → 完整模型 ====================
# 训练完成后执行；合并后 ≈ 3.8GB，部署时无需 peft（merge 即 W ← W + α/r·BA）
base_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)   # 也可换成某个 checkpoint-XX 目录
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("已导出:", MERGED_DIR)

Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.83s/it]

已导出: ./lora_merged


In [12]:
# ==================== 12)（可选）推理演示：先思考、后回答 ====================
# 训练完成后运行；直接用带适配器的 lora_model 生成
lora_model.eval()
lora_model.config.use_cache = True   # 训练时被梯度检查点关掉，推理时开回来（加速）

question = "一名 55 岁男性，近一周反复头晕，自测血压 160/100 mmHg，应该怎么处理？"
messages = [
    {"role": "system", "content": "您是一位在临床推理、诊断和治疗计划方面具有高级知识的医学专家。请回答以下医学问题。"},
    {"role": "user", "content": question},
]
# 推理时同样给模型开思考脚手架（enable_thinking=True → 结尾 "<think>\n"，模型接着写思考过程）
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, enable_thinking=True,
    tokenize=True, return_tensors="pt", return_dict=True,
).to(lora_model.device)

out = lora_model.generate(**inputs, max_new_tokens=1024)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

好的，我现在需要处理这个55岁男性患者的情况。首先，他最近一周反复头晕，自测的血压是160/100 mmHg。这看起来有点高，但需要仔细分析。

首先，我得考虑高血压的分级。根据中国高血压防治指南，收缩压160 mmHg属于2级高血压，而舒张压100 mmHg也属于2级。所以，这个患者的血压确实偏高，需要进一步评估。

接下来，头晕的症状可能和高血压有关，但也可能是其他原因，比如脑血管问题。这时候，我应该先排除其他可能的原因，比如是否有其他症状，比如头痛、视力模糊，或者是否有其他疾病，如糖尿病、高血脂等。

然后，我需要考虑患者的年龄和病史。55岁属于中年，可能有高血压的家族史，或者是否有其他风险因素，比如吸烟、肥胖、缺乏运动等。这些都会增加高血压的风险。

另外，头晕的症状持续一周，说明可能已经持续了一段时间，需要尽快处理。这时候，应该建议患者尽快就医，进行进一步的检查，比如心电图、心脏超声，或者脑部CT/MRI，以排除其他潜在问题。

同时，患者应该被建议进行生活方式的调整，比如低盐饮食、控制体重、减少咖啡因摄入，以及增加有氧运动。这些措施可以帮助控制血压，减少头晕的发生。

此外，如果患者有糖尿病或高血脂，可能需要进一步检查血糖和血脂水平，因为这些因素也可能导致高血压。

最后，如果血压持续偏高，可能需要药物治疗。这时候，应该根据患者的具体情况，选择合适的降压药，并定期监测血压，确保血压控制在目标范围内。

总之，处理这个患者的情况需要综合考虑血压水平、头晕症状、年龄、病史和其他风险因素，通过进一步检查、生活方式调整、可能的药物治疗来综合管理。
</think>

### 分析步骤：
1. **血压评估**：收缩压160 mmHg和舒张压100 mmHg属于2级高血压，需进一步评估。
2. **头晕症状**：持续一周的头晕可能提示高血压相关症状，但也需排除其他潜在问题。
3. **风险因素**：55岁年龄可能增加高血压风险，需考虑是否有其他疾病（如糖尿病、高血脂）或家族史。
4. **初步建议**：
   - 建议尽快就医，进行进一步检查（如心电图、心脏超声、脑部CT/MRI）。
   - 生活方式调整：低盐饮食、控制体重、减少咖啡因摄入、增加有氧运动。
   - 若存在糖尿病或高血脂，需进一步检查血糖和血脂水平。
5. **药物治疗**：若血压持续偏高，需根

In [14]:
# ==================== 13)（可选）基座模型 vs 微调后：同题对照 ====================
# 同一个问题分别问「基座模型（未微调）」与「LoRA 微调后（合并模型）」，直接观察行为对齐差异
# 注意：6GB 显存装不下两个模型 → 依次"加载→生成→删除→清缓存"；本格自带加载，清理后也能直接跑（每次生成约几分钟）

import gc, time

question = "一名 55 岁男性，近一周反复头晕，自测血压 160/100 mmHg，应该怎么处理？"   # 与推理格相同的问题
messages = [
    {"role": "system", "content": "您是一位在临床推理、诊断和治疗计划方面具有高级知识的医学专家。请回答以下医学问题。"},
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, enable_thinking=True,
    tokenize=True, return_tensors="pt", return_dict=True)

def ask_once(model_path):
    model_ = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16).to("cuda")
    model_.eval()
    t0 = time.time()
    out = model_.generate(**inputs.to(model_.device), max_new_tokens=1024)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    secs = time.time() - t0
    del model_, out
    gc.collect(); torch.cuda.empty_cache()      # 先释放，再加载下一个（6GB 显存）
    return text, secs

answer_base, t_base = ask_once(MODEL_PATH)      # 基座（未微调）
answer_lora, t_lora = ask_once(MERGED_DIR)      # LoRA 微调后（合并模型）

print("=" * 70)
print(f"【基座模型（未微调）】用时 {t_base/60:.1f} 分钟")
print("=" * 70)
print(answer_base)
print()
print("=" * 70)
print(f"【LoRA 微调后（合并模型）】用时 {t_lora/60:.1f} 分钟")
print("=" * 70)
print(answer_lora)

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 3400.03it/s]


【基座模型（未微调）】用时 0.8 分钟
Here's a thinking process that leads to the suggested response:

1.  **Analyze the User's Request:**
    *   **Patient:** 55-year-old male.
    *   **Symptoms:** Recurrent dizziness for the past week.
    *   **Vital Sign:** Self-measured blood pressure 160/100 mmHg.
    *   **Question:** How to handle/treat this situation?
    *   **Role:** Medical expert (clinical reasoning, diagnosis, treatment planning).
    *   **Constraint:** Provide medical advice based on clinical reasoning, but *must* include a disclaimer that I am an AI and this is not a substitute for professional medical advice.

2.  **Initial Assessment & Risk Stratification:**
    *   **BP:** 160/100 mmHg. This is Stage 2 Hypertension (according to JNC 8/2017/2018 guidelines, specifically >140/90 for Stage 1, >160/100 for Stage 2). It's definitely hypertensive.
    *   **Symptom:** Dizziness. This is a symptom of hypoperfusion or vascular issues.
    *   **Duration:** 1 week. This is acute/subacute.
 

In [13]:
# ==================== 14)（可选）释放显存 / 内存 ====================
# 全部实验结束后可执行本格；也可不执行——重启/关闭内核时这些内存会自动全部释放
# 注意：本格会删除本会话中的模型对象（想再用模型需重跑前面的格）
import gc

print("清理前 : allocated {:.2f} GB / reserved {:.2f} GB".format(
    torch.cuda.memory_allocated()/1024**3, torch.cuda.memory_reserved()/1024**3))

del trainer, lora_model, model, base_model, merged_model
gc.collect()                # 回收 Python 对象（RAM）
torch.cuda.empty_cache()    # 把 PyTorch 缓存池中的空闲显存还给驱动

print("清理后 : allocated {:.2f} GB / reserved {:.2f} GB".format(
    torch.cuda.memory_allocated()/1024**3, torch.cuda.memory_reserved()/1024**3))
print("模型文件都在磁盘（results_lora / lora_adapter / lora_merged），可随时重载")

清理前 : allocated 3.53 GB / reserved 19.04 GB
清理后 : allocated 0.02 GB / reserved 0.03 GB
模型文件都在磁盘（results_lora / lora_adapter / lora_merged），可随时重载


## 实验结果与分析

### 1. 训练指标

| Step | Train Loss | Eval Loss |
|---|---|---|
| 10 | 2.370 | 2.288 |
| 20 | 2.251 | 2.229 |
| 25 | 2.237 | **2.226**（最优） |

- 全程 **25/25 步、24:51**（含 3 次评估；batch 1 × 累积 32，等效 batch 32）
- Eval Loss 从 2.2877 降到 2.2258（token 级困惑度 $e^{2.23} \approx 9.3$），且**到第 25 步仍在下降、train/eval 差距很小**——160 条 × 5 轮还没训饱，**增加数据量 / 轮数仍有明显收益，暂不用担心过拟合**
- `load_best_model_at_end` 生效：最优 = `checkpoint-25`

### 2. 参数效率（LoRA 的核心收益）

| 项目 | 全参微调（1.88B） | 本实验 LoRA |
|---|---|---|
| 可训练参数 | 18.8 亿（100%） | **73.7 万（0.039%）** |
| 优化器状态（Adam fp32 两份动量） | ≈ 15 GB | ≈ **6 MB** |
| 产物大小 | 整个模型 ≈ 3.5 GB | 适配器 **2.8 MB**（约基座的 0.08%） |
| 部署 | 直接可用 | `merge_and_unload()` 后与原模型无差别，零额外推理开销 |

- 24 个挂载点（6 个全注意力层 × q/k/v/o）；显存大头始终是**基座权重（bf16 ≈ 3.8GB）**，LoRA 自身开销可忽略
- 适配器只有 2.8MB，天然适合「一个基座 + 多个任务适配器」的分发方式

### 3. 显存与工程记录（6GB 卡踩坑复盘）

1. **评估 OOM**：评估 batch 默认 8，248k 词表的 logits 在交叉熵里上采 fp32 单块需 ~6.35GB——训练正常、评估即崩。修复：`per_device_eval_batch_size=1`
2. **Windows 共享显存溢出**：训练时 `reserved` 高达 **19GB**（远超 6GB 物理显存），大量数据溢出到系统内存 → 约 1 分钟/步（比理论速度慢）
3. **内存不自动释放**：训练结束后对象仍被引用 + PyTorch 缓存分配器保留内存块。清理实测：allocated **3.53 GB → 0.02 GB**，GPU 占用 **5951 → 543 MiB**，进程 RAM **8.8 → 0.7 GB**

### 4. 推理效果（行为对齐证据）

用同一个高血压病例测试微调后的生成：

- ✅ **学会了「先思考、后回答」**：先输出一大段推理（血压分级 → 眩晕鉴别 → 风险因素 → 处理思路），以 `</think>` 收口，再给出「### 分析步骤 + 总结」的结构化正式答案——这正是本次 SFT 追求的行为对齐（格式 + 解题模式），而非单纯知识注入
- ✅ **内容基本正确**：160/100 mmHg 判为 2 级高血压（符合中国指南分级），处理建议（就医检查 / 生活方式 / 必要时药物）方向合理
- ⚠️ **不足**：思考略啰嗦、深度一般——与"只训了 25 步"的欠训状态一致，加量后预期明显改善

**基座 vs 微调后对照**（对照格输出：同一问题、同一生成参数）：

| 维度 | 基座（未微调） | LoRA 微调后 |
|---|---|---|
| 思考语言 | **英文**（`Here's a thinking process…`，引用 JNC 8 等英文指南） | **中文**（「根据中国高血压防治指南…」） |
| 思考风格 | 罗列 8 大步骤的「元思考」，偏啰嗦 | 紧凑递进：分级 → 鉴别 → 风险 → 处理 |
| 输出完整性 | 1024 token 内没写完、**被截断**（还没进入正式答案） | 完整收口：中文思考 + 结构化答案 |

→ 训练数据是**中文 CoT**，微调让模型从基座的「英文思考」切换为「中文思考 + 中文回答」——**语言与格式的切换本身就是行为对齐的直接证据**。

### 5. 结论

对应实验目的逐条核对：

1. ✅ LoRA 原理与超参数：r=8 / α=16，只训 0.039% 参数完成微调
2. ✅ peft 版 SFT 全流程闭环：数据 → 模板 → 掩码 → 注入 → 训练 → 保存 → 合并导出
3. ✅ 训练样本构造：160/160 前缀断言通过，只在 assistant 回答段计算 loss
4. ✅ 效果观察：学会「先思考、后回答」+ 结构化输出；对照显示思考语言也从英文对齐为中文
5. ✅ 产物管理：checkpoint（28MB/个）→ best → adapter（2.8MB）→ merged（3.5GB），各自用途均已验证

**当前瓶颈是数据量 / 步数，而不是 LoRA 容量**（eval 仍在下降）。下一阶段实验：① 对照扩展：多跑几组问题，或加入「请用中文思考并回答」的提示约束再对照（检验语言控制）；② 加大子集到 1000~2000 条、训练 10+ 轮；③ 消融 r/α，或把 `linear_attn` / MLP 加入 `target_modules`。